# AraStudy Exp02 (Colab Parallel Runner)
Run one tokenizer/seed job at a time, persist artifacts to Drive, and safely resume after disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content
!rm -rf arastudy
!git clone https://github.com/faresrafat3/arastudy
%cd /content/arastudy
!pip install -r requirements.txt -q

In [ ]:
import os
import zipfile

zip_path = "/content/drive/MyDrive/arastudy_phase2b_data.zip"
extract_root = "/content/drive/MyDrive/arastudy_data"

os.makedirs(extract_root, exist_ok=True)

if not os.path.exists(f"{extract_root}/splits/phase2b/valid.txt"):
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_root)
    print("✅ Extracted zip to Drive")
else:
    print("✅ Data already extracted")

!mkdir -p data/splits results/tokenizers
!ln -sfn /content/drive/MyDrive/arastudy_data/splits/phase2b data/splits/phase2b
!ln -sfn /content/drive/MyDrive/arastudy_data/tokenizers/phase2b results/tokenizers/phase2b

!ls -lah data/splits/phase2b | head -n 10
!ls -lah results/tokenizers/phase2b | head -n 20
!test -f data/splits/phase2b/valid.txt && echo OK_VALID || echo MISSING_VALID

In [ ]:
TOKENIZER = "bpe_16k"
SEED = 123
run_id = f"exp02_{TOKENIZER}_s{SEED}"
print("RUN_ID =", run_id)

!python -m src.training.train_exp01_full \
  --config configs/experiments/exp02_multiseed.yaml \
  --tokenizer-id {TOKENIZER} \
  --seed {SEED} \
  --run-id {run_id}

In [ ]:
import os
import shutil

src_dir = f"results/logs/exp02/{run_id}"
dst_dir = f"/content/drive/MyDrive/arastudy_results/exp02/{run_id}"
os.makedirs(os.path.dirname(dst_dir), exist_ok=True)
shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)
print(f"✅ Saved to {dst_dir}")

In [ ]:
!ls -lah /content/drive/MyDrive/arastudy_results/exp02/{run_id}
!test -f /content/drive/MyDrive/arastudy_results/exp02/{run_id}/{TOKENIZER}_summary.json && echo OK_SUMMARY || echo MISSING_SUMMARY